# **RNN과 자연어처리**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **3. RNN · LSTM · GRU**로 문장 읽기

| Lab | 주제 | 발전사 위치 |
|---|---|---|
| 3-1 |  RNN 셀 손구현 = `nn.RNN` | 1990 Elman |
| 3-2 |기울기 소실을 눈으로 보기: RNN vs LSTM vs GRU | 1991 Hochreiter · 1994 Bengio |
| 3-3 | NSMC 감성 분류: TF-IDF 기준선 vs RNN·LSTM·GRU·BiLSTM | 1997 LSTM · 2014 GRU |
| 3-4 | 4차시 감정 6분류 데이터를 LSTM으로 | 다중 분류 |
| 3-5 | 데이터가 적을 때: Word2Vec 사전학습 임베딩의 힘 | 2013~2018 사전학습의 시작 |

In [ ]:
%pip install -q kiwipiepy koreanize-matplotlib gensim

In [ ]:
import os, urllib.request, time, pickle, json, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False   # 음수 기호 깨짐 방지
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
os.makedirs("figs", exist_ok=True)

## **3-0. MLP·CNN은 왜 순서 데이터에 약한가**
같은 한국어 문장으로 MLP vs CNN vs RNN 비교


In [ ]:
# -*- coding: utf-8 -*-
"""
[실습] MLP·CNN은 왜 순서 데이터에 약한가 — 같은 한국어 문장으로 MLP vs CNN vs RNN 비교

과제: "A는 ~라고 생각했다 하지만 B는 ~라고 생각했다"에서 '하지만' 뒤의 평가가 최종 감성이다.
  - 모든 문장에 긍정 단어 1개 + 부정 단어 1개가 함께 들어 있다.
  - 정답은 '어떤 단어가 있나'가 아니라 '어떤 순서로 나오나'로만 정해진다.

슬라이드의 세 약점을 실험으로 확인한다.
  ① 입력 길이가 고정된다         → MLP: 칸 수보다 긴 문장은 뒤가 잘린다        (실험 B)
  ② 위치마다 따로 배운다         → MLP: 같은 문장도 단어 위치가 밀리면 흔들린다  (실험 C)
  ③ 멀리 떨어진 앞 내용을 놓친다 → CNN: 창(3단어) 밖의 단서를 묶지 못한다       (실험 A)
실행: pip install torch
"""
import random, time
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

# ─────────────────────────────────────────────
# 1) 데이터: 같은 단어, 다른 순서 → 다른 감성
#    (띄어쓰기로 조사·어미를 떼어 놓은 형태소 분석 결과처럼 표현)
# ─────────────────────────────────────────────
TOPIC = ["영화", "배우", "스토리", "음악", "연출", "결말", "영상미", "대사"]
POS   = ["재밌다", "최고다", "감동적이다", "훌륭하다", "신선하다"]
NEG   = ["지루하다", "별로다", "실망이다", "아쉽다", "뻔하다"]
INTEN = ["정말", "너무", "진짜", "꽤", "좀", "엄청"]
# 한 문장 안에서는 앞 절과 뒤 절이 '같은' 머리·꼬리를 쓴다
# → 감성 단어 바로 옆 단어만 봐서는 앞 절인지 뒤 절인지 알 수 없게 만든다
HEAD = [[], ["처음엔"], ["솔직히"], ["직접", "보니"], ["끝까지", "보고", "나니"]]
TAIL = [[], ["고", "생각했다"], ["는", "평이", "많았다"], ["고", "다들", "말했다"],
        ["라는", "느낌이", "강하게", "들었다"], ["고", "친구가", "말했다"]]

def josa(word):
    """받침 있으면 '은', 없으면 '는' (한글 음절: (코드-0xAC00) % 28 == 0 이면 받침 없음)"""
    return "은" if (ord(word[-1]) - 0xAC00) % 28 else "는"

def clause(senti, head, tail, n_inten):
    """절 하나 = [머리] 주제 은/는 [강조어] 감성단어 [꼬리]"""
    topic = random.choice(TOPIC)
    pre = head + [topic, josa(topic)] + random.choices(INTEN, k=n_inten)
    return pre + [senti] + tail, len(pre)

def make_sentence(label, max_inten=2):
    """label=1: 하지만 뒤가 긍정 / label=0: 하지만 뒤가 부정"""
    s1, s2 = (random.choice(NEG), random.choice(POS)) if label == 1 else \
             (random.choice(POS), random.choice(NEG))
    head, tail = random.choice(HEAD), random.choice(TAIL)
    c1, _    = clause(s1, head, tail, random.randint(0, max_inten))
    c2, pre2 = clause(s2, head, tail, random.randint(0, max_inten))
    toks = c1 + ["하지만"] + c2
    # 실험 A용: 감성 단어와 '하지만'이 한 창에 함께 들어오려면 필요한 최소 거리
    reach = min(len(tail) + 1,   # 앞 감성 단어 → 하지만
                pre2 + 1)        # 하지만 → 뒤 감성 단어
    return " ".join(toks), label, reach

def make_data(n, **kw):
    return [make_sentence(i % 2, **kw) for i in range(n)]

N_TRAIN = 300                      # ★ 데이터가 적을수록 '위치마다 따로 배우는' MLP가 불리하다
train_data = make_data(N_TRAIN)
test_data  = make_data(1000)
print("예시 문장")
for s, y, _ in train_data[:4]:
    print(f"  [{'긍정' if y else '부정'}] {s}")

# 토큰 → 번호 (0은 패딩)
vocab = {"<pad>": 0}
for s, _, _ in train_data:
    for w in s.split():
        vocab.setdefault(w, len(vocab))
SLOTS = max(len(s.split()) for s, _, _ in train_data)   # ① MLP의 고정 칸 수 = 학습 문장 최대 길이

def encode(data):
    """가변 길이 인코딩: 배치 안 가장 긴 문장에 맞춰 0으로 채운다 (자르지 않음)"""
    T = max(len(s.split()) for s, _, _ in data)
    X = torch.zeros(len(data), T, dtype=torch.long)
    L = torch.zeros(len(data), dtype=torch.long)
    for i, (s, _, _) in enumerate(data):
        ids = [vocab[w] for w in s.split()]
        X[i, :len(ids)] = torch.tensor(ids); L[i] = len(ids)
    y = torch.tensor([lab for _, lab, _ in data], dtype=torch.float)
    return X, L, y

Xtr, Ltr, ytr = encode(train_data)
print(f"\n어휘 수 {len(vocab)}, MLP 고정 칸 수 {SLOTS}, 학습 {len(Xtr)} / 테스트 {len(test_data)}")

# ─────────────────────────────────────────────
# 2) 세 모델: 임베딩(32차원)은 같고, 문장을 '읽는 방식'만 다르다
# ─────────────────────────────────────────────
EMB, HID = 32, 64

class MLPText(nn.Module):
    """① 고정 칸 + ② 칸마다 다른 가중치: 단어 벡터를 칸 순서대로 이어 붙여 한 번에 입력"""
    def __init__(self, V):
        super().__init__()
        self.emb = nn.Embedding(V, EMB, padding_idx=0)
        self.fc  = nn.Sequential(nn.Flatten(),                 # (B, SLOTS, E) → (B, SLOTS*E)
                                 nn.Linear(SLOTS * EMB, HID),  # W1, W2, …, W_SLOTS 가 칸마다 다름
                                 nn.ReLU(), nn.Linear(HID, 1))
    def forward(self, x, lengths):
        x = x[:, :SLOTS]                                       # ① 칸보다 길면 뒤가 잘림
        if x.size(1) < SLOTS:                                  # ① 짧으면 0으로 채움
            x = nn.functional.pad(x, (0, SLOTS - x.size(1)))
        return self.fc(self.emb(x)).squeeze(1)

class CNNText(nn.Module):
    """③ 창(3단어)을 밀며 지역 패턴을 찾고, 전역 최대 풀링으로 문장 전체를 요약 (Kim, 2014)"""
    def __init__(self, V, k=3):
        super().__init__()
        self.emb  = nn.Embedding(V, EMB, padding_idx=0)
        self.conv = nn.Conv1d(EMB, 96, kernel_size=k)          # 같은 필터를 모든 위치에 공유
        self.fc   = nn.Linear(96, 1)
    def forward(self, x, lengths):
        e = self.emb(x).transpose(1, 2)                        # (B, E, T) Conv1d 입력 형태
        h = torch.relu(self.conv(e)).max(dim=2).values         # 어디서 나왔는지(위치)는 버림
        return self.fc(h).squeeze(1)

class RNNText(nn.Module):
    """길이에 상관없이, 같은 가중치로, 앞 내용을 은닉 상태에 기억하며 한 단어씩 읽는다"""
    def __init__(self, V):
        super().__init__()
        self.emb  = nn.Embedding(V, EMB, padding_idx=0)
        self.lstm = nn.LSTM(EMB, HID, batch_first=True)
        self.fc   = nn.Linear(HID, 1)
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(            # 패딩은 읽지 않음
            self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.fc(h[-1]).squeeze(1)                       # 마지막 단어까지 읽은 기억

# ─────────────────────────────────────────────
# 3) 같은 조건으로 학습
# ─────────────────────────────────────────────
def train(model, epochs=40, bs=32, lr=5e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            loss = lossf(model(Xtr[idx], Ltr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
    return time.time() - t0

@torch.no_grad()
def predict(model, data):
    model.eval()
    X, L, y = encode(data)
    return torch.sigmoid(model(X, L)), y

def acc(model, data):
    p, y = predict(model, data)
    return ((p > 0.5).float() == y).float().mean().item()

models = {}
print(f"\n{'모델':10s} | {'파라미터':>7s} | {'학습시간':>6s} | {'학습 정확도':>7s} | {'테스트 정확도':>7s}")
for name, cls in [("MLP", MLPText), ("CNN", CNNText), ("RNN(LSTM)", RNNText)]:
    torch.manual_seed(SEED)
    m = cls(len(vocab))
    sec = train(m)
    models[name] = m
    n_par = sum(p.numel() for p in m.parameters())
    print(f"{name:10s} | {n_par:9,d} | {sec:6.1f}초 | {acc(m, train_data):10.1%} | {acc(m, test_data):12.1%}")

header = f"{'':20s} | " + " | ".join(f"{n:>9s}" for n in models)
def row(title, data):
    print(f"{title:20s} | " + " | ".join(f"{acc(m, data):9.1%}" for m in models.values()))

# ─────────────────────────────────────────────
# 실험 A — ③ 창 밖의 단서: 감성 단어와 '하지만' 사이 거리별 정확도
# ─────────────────────────────────────────────
print("\n[실험 A] ③ 거리별 정확도 — CNN 창은 3단어라 거리 2까지만 한 창에 들어온다")
print("         (거리 1 예: '영화 는 재밌다 하지만 …' → '재밌다 하지만'이 한 창에 들어옴)")
print(header)
for d in sorted({r for _, _, r in test_data}):
    sub = [t for t in test_data if t[2] == d]
    row(f"거리 {d} (n={len(sub)})", sub)

# ─────────────────────────────────────────────
# 실험 B — ① 고정 길이: 학습 때보다 긴 문장 (강조어를 최대 5개까지)
# ─────────────────────────────────────────────
random.seed(SEED + 1)
long_data = [t for t in make_data(3000, max_inten=5) if len(t[0].split()) > SLOTS][:500]
print(f"\n[실험 B] ① 칸 수({SLOTS})보다 긴 문장 {len(long_data)}개 — MLP는 문장 끝(결론)이 잘린다")
print(header)
row("긴 문장", long_data)

# ─────────────────────────────────────────────
# 실험 C — ② 위치 이동: 같은 문장 앞에 '정말'을 k번 붙여 단어 위치만 민다 (뜻은 그대로)
# ─────────────────────────────────────────────
print("\n[실험 C] ② 같은 테스트 문장을 오른쪽으로 k칸 밀기 (칸 수 안에 들어오는 문장만)")
print(header)
for k in (0, 1, 2, 3):
    shifted = [(" ".join(["정말"] * k + s.split()), y, r) for s, y, r in test_data]
    shifted = [t for t in shifted if len(t[0].split()) <= SLOTS]   # 잘림 효과는 빼고 위치만 비교
    row(f"{k}칸 이동 (n={len(shifted)})", shifted)

# ─────────────────────────────────────────────
# 반전 문장 쌍: 단어는 같고 순서만 바꿈
# ─────────────────────────────────────────────
pairs = [
    ("처음엔 영화 는 정말 지루하다 고 생각했다 하지만 처음엔 결말 은 너무 재밌다 고 생각했다", 1),
    ("처음엔 영화 는 정말 재밌다 고 생각했다 하지만 처음엔 결말 은 너무 지루하다 고 생각했다", 0),
    ("끝까지 보고 나니 배우 는 별로다 는 평이 많았다 하지만 끝까지 보고 나니 음악 은 꽤 훌륭하다 는 평이 많았다", 1),
    ("끝까지 보고 나니 배우 는 훌륭하다 는 평이 많았다 하지만 끝까지 보고 나니 음악 은 꽤 별로다 는 평이 많았다", 0),
]
probs = {n: predict(m, [(s, y, 0) for s, y in pairs])[0] for n, m in models.items()}
print("\n[반전 문장 쌍] 단어는 같고 순서만 다름 — 긍정 확률 (0.5 근처 = 구분 못 함)")
print(f"{'정답':4s} | " + " | ".join(f"{n:>9s}" for n in models) + " | 문장")
for i, (s, y) in enumerate(pairs):
    print(f"{'긍정' if y else '부정':4s} | " + " | ".join(f"{probs[n][i].item():9.2f}" for n in models) + f" | {s}")

---
## **3-1. RNN 셀 손구현** = `nn.RNN`
$h_t = \tanh(W_{x} x_t + b_{x} + W_{h} h_{t-1} + b_{h})$ — **같은 가중치**를 모든 시점에서 재사용한다.

In [ ]:
D, H, T = 3, 4, 5                     # 입력 차원, 은닉 차원, 길이
rnn = nn.RNN(D, H, batch_first=True)
x = torch.randn(1, T, D)

h = torch.zeros(H)
for t in range(T):                    # 반복문으로 시간 축을 펼친다
    h = torch.tanh(rnn.weight_ih_l0 @ x[0, t] + rnn.bias_ih_l0 + rnn.weight_hh_l0 @ h + rnn.bias_hh_l0)
out, h_n = rnn(x)
print("손구현 h_T :", h.detach().numpy().round(4))
print("nn.RNN h_T:", h_n[0, 0].detach().numpy().round(4))
print("같은가?", torch.allclose(h, h_n[0, 0], atol=1e-6))

---
## **3-2. 기울기 소실 실험** — 마지막 출력의 기울기가 과거 입력까지 얼마나 전달될까?
$\dfrac{\partial h_T}{\partial h_t} = \prod_{k=t+1}^{T} \dfrac{\partial h_k}{\partial h_{k-1}}$ — 곱셈이 길게 이어지면 0으로 사라지거나(소실) 폭발한다.

In [ ]:
def grad_profile(cell, T=100, H=64, D=16, trials=10, keep_bias=0.0):
    norms = np.zeros(T)
    for s in range(trials):
        torch.manual_seed(s)
        m = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[cell](D, H, batch_first=True)
        if keep_bias:                                   # '기억 유지' 쪽으로 게이트를 열어 둔 초기화
            with torch.no_grad():
                if cell == "LSTM": m.bias_ih_l0[H:2*H].fill_(keep_bias)   # 게이트 순서 (i, f, g, o) → 망각 게이트 f
                if cell == "GRU":  m.bias_ih_l0[H:2*H].fill_(keep_bias)   # 게이트 순서 (r, z, n) → 업데이트 게이트 z
        x = torch.randn(1, T, D, requires_grad=True)
        out, _ = m(x)
        out[0, -1].sum().backward()          # 마지막 시점 출력만으로 손실을 만든다
        norms += x.grad[0].norm(dim=1).numpy()
    return norms / trials

plt.figure(figsize=(8.5, 4.4))
for (cell, kb, lab, c, ls) in [("RNN", 0, "RNN", "#DC2626", "-"),
                               ("LSTM", 0, "LSTM (기본 초기화)", "#004A80", "--"),
                               ("GRU", 0, "GRU (기본 초기화)", "#0EA5B7", "--"),
                               ("LSTM", 3.0, "LSTM (망각 게이트 bias=3)", "#004A80", "-"),
                               ("GRU", 3.0, "GRU (업데이트 게이트 bias=3)", "#0EA5B7", "-")]:
    g = grad_profile(cell, keep_bias=kb)
    plt.semilogy(range(1, 101), g, ls, label=lab, color=c, lw=2.2)
    print(f"{lab:<24} t=100(최근) {g[-1]:.1e}  t=50 {g[49]:.1e}  t=1(가장 먼 과거) {g[0]:.1e}")
plt.xlabel("입력 시점 t  (100 = 마지막)"); plt.ylabel("‖∂출력/∂x_t‖ (로그 축)")
plt.title("마지막 출력의 기울기는 과거 입력까지 얼마나 전달되는가 (T=100, 학습 전)")

from matplotlib.ticker import FuncFormatter
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"1e{int(round(np.log10(v)))}"))   # 10^-3 → 1e-3 (한글 폰트 음수 기호 문제 회피)
plt.legend(fontsize=9)
plt.grid(alpha=.3, which="both")
plt.tight_layout()
plt.savefig("figs/grad_vanish.png", dpi=160)
plt.show()

**관찰**: 게이트가 있어도 기본 초기화에서는 기울기가 줄어든다. 망각(업데이트) 게이트를 "기억 유지" 쪽으로 열어 두면 100칸 전까지 기울기가 살아 있다
→ LSTM의 핵심은 **덧셈으로 이어지는 셀 상태 + 학습으로 여닫는 게이트** (Gers et al., 2000; Jozefowicz et al., 2015: forget bias를 1로 초기화 권장)

---
## **3-3. NSMC 감성 분류** — RNN 계열 4종 vs TF-IDF 기준선
파이프라인: 문장 → **Kiwi 형태소** → 정수 ID → `nn.Embedding` → RNN/LSTM/GRU → 마지막 은닉 상태 → `nn.Linear`

In [ ]:
URLS = {
    "ratings_train.txt": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "ratings_test.txt":  "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    "steam.txt":         "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt",
}
for fn, url in URLS.items():
    if not os.path.exists(fn):
        urllib.request.urlretrieve(url, fn)
train = pd.read_csv("ratings_train.txt", sep="\t").dropna()
test  = pd.read_csv("ratings_test.txt",  sep="\t").dropna()
steam = pd.read_csv("steam.txt", sep="\t", header=None, names=["label", "document"], quoting=3).dropna().sample(20000, random_state=42)

from kiwipiepy import Kiwi
kiwi = Kiwi()
def kiwi_tok(texts, cache):
    if os.path.exists(cache):
        return pickle.load(open(cache, "rb"))
    toks = [[t.form for t in r] for r in kiwi.tokenize(texts)]
    pickle.dump(toks, open(cache, "wb"))
    return toks
t0 = time.time()
tok_tr = kiwi_tok(train.document.tolist(), "tok_train.pkl")
tok_te = kiwi_tok(test.document.tolist(), "tok_test.pkl")
tok_st = kiwi_tok(steam.document.tolist(), "tok_steam20k.pkl")
print(f"형태소 분석 {time.time()-t0:.0f}초")

In [ ]:
from collections import Counter
MAX_LEN = 50
cnt = Counter(w for s in tok_tr for w in s)                        # 어휘는 train에서만 만든다 (누수 방지)
itos = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= 3]
stoi = {w: i for i, w in enumerate(itos)}

def encode(toks):
    ids = [[stoi.get(w, 1) for w in s][:MAX_LEN] or [1] for s in toks]
    lens = torch.tensor([len(s) for s in ids])
    X = torch.zeros(len(ids), MAX_LEN, dtype=torch.long)
    for i, s in enumerate(ids):
        X[i, :len(s)] = torch.tensor(s)
    return X, lens

Xtr, Ltr = encode(tok_tr); ytr = torch.tensor(train.label.values)
Xte, Lte = encode(tok_te); yte = torch.tensor(test.label.values)
Xst, Lst = encode(tok_st); yst = torch.tensor(steam.label.values)
print(f"어휘 {len(itos):,}개 | 학습 {Xtr.shape} | 평균 길이 {Ltr.float().mean():.1f} 토큰")

In [ ]:
class TextRNN(nn.Module):
    def __init__(self, vocab, cell="lstm", emb=100, hidden=128, bi=False, n_cls=2, pretrained=None):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        if pretrained is not None:
            self.emb.weight.data.copy_(pretrained)                  # Lab 3-4: 사전학습 임베딩으로 초기화
        RNN = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn, self.cell, self.bi = RNN(emb, hidden, batch_first=True, bidirectional=bi), cell, bi
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden * (2 if bi else 1), n_cls)
    def forward(self, x, lens):
        packed = pack_padded_sequence(self.emb(x), lens.cpu(), batch_first=True, enforce_sorted=False)  # <pad>는 읽지 않는다
        _, h = self.rnn(packed)
        if self.cell == "lstm":
            h = h[0]                                                  # LSTM은 (h, c) 중 h 사용
        last = torch.cat([h[-2], h[-1]], dim=1) if self.bi else h[-1]  # 양방향이면 정방향+역방향 이어붙이기
        return self.fc(self.drop(last))

@torch.no_grad()
def evaluate(model, X, L, y, bs=1024):
    model.eval(); pred = []
    for i in range(0, len(X), bs):
        pred.append(model(X[i:i+bs].to(device), L[i:i+bs]).argmax(1).cpu())
    return (torch.cat(pred) == y).float().mean().item()

def fit(model, X, L, y, epochs=3, bs=256, lr=2e-3, verbose=True):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(X)); t0 = time.time()
        for i in range(0, len(X), bs):
            idx = perm[i:i+bs]
            loss = nn.functional.cross_entropy(model(X[idx].to(device), L[idx]), y[idx].to(device))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)          # 기울기 폭발 방지 (클리핑)
            opt.step()
        if verbose:                                                   # 에폭마다 테스트 정확도 기록
            acc = evaluate(model, Xte, Lte, yte); hist.append(acc)
            print(f"  epoch {ep+1}: test acc {acc:.4f} ({time.time()-t0:.0f}s)")
    return hist

In [ ]:
# ⏱ CPU 기준 모델당 약 2~4분. 시간이 부족하면 N_TRAIN을 50000으로 줄여도 흐름은 같다.
N_TRAIN = len(Xtr)
configs = {"RNN": dict(cell="rnn"), "LSTM": dict(cell="lstm"), "GRU": dict(cell="gru"), "BiLSTM": dict(cell="lstm", bi=True)}
res, models, curves = {}, {}, {}
for name, cfg in configs.items():
    seed_all(42); print(name)
    t0 = time.time()
    m = TextRNN(len(itos), **cfg)
    curves[name] = fit(m, Xtr[:N_TRAIN], Ltr[:N_TRAIN], ytr[:N_TRAIN])
    n_param = sum(p.numel() for p in m.parameters())
    res[name] = dict(test=curves[name][-1], steam=evaluate(m, Xst, Lst, yst), params=n_param, sec=round(time.time()-t0))
    models[name] = m

In [ ]:
base = json.load(open("baseline_tfidf.json")) if os.path.exists("baseline_tfidf.json") else {"TF-IDF 형태소 1~2gram": 0.8695, "Steam(도메인 이동)": 0.7224}
rows = [["TF-IDF + 로지스틱 회귀 ", base["TF-IDF 형태소 1~2gram"], base["Steam(도메인 이동)"], "—", "—"]]
rows += [[k, v["test"], v["steam"], f'{v["params"]:,}', v["sec"]] for k, v in res.items()]
result_table = pd.DataFrame(rows, columns=["모델", "NSMC 테스트 정확도", "Steam 정확도(도메인 이동)", "파라미터 수", "학습 시간(초)"])
result_table.to_csv("rnn_results.csv", index=False)
result_table.round(4)

In [ ]:
# 어순·부정 표현 테스트: BoW()는 틀렸던 문장
best = models["BiLSTM"]
def predict(m, sents):
    X, L = encode([[t.form for t in kiwi.tokenize(s)] for s in sents])
    m.eval()
    with torch.no_grad():
        return torch.softmax(m(X.to(device), L), 1)[:, 1].cpu().numpy()
probe = ["재미 없는 건 아니고 진짜 최고", "진짜 최고 아니고 재미 없는 건",
         "처음엔 지루했는데 끝까지 보니 인생 영화", "처음엔 재밌었는데 끝까지 보니 시간 낭비",
         "배우 연기는 좋은데 스토리가 별로"]
probe_df = pd.DataFrame({"문장": probe, "긍정 확률(BiLSTM)": predict(best, probe).round(3)})
probe_df.to_csv("probe.csv", index=False)
probe_df

---
## **3-4. 감정 6분류 데이터(영어 트윗)를 LSTM으로**
4차시에서 사용한 `multi-class-train.txt` / `multi-class-val.txt` (Saravia et al., 2018, CARER)를 그대로 쓴다.

In [ ]:
def load_emotion(fn):
    return pd.read_csv(fn, sep=";", header=None, names=["text", "label"])
if os.path.exists("multi-class-train.txt"):
    emo_tr = load_emotion("multi-class-train.txt")
    if os.path.exists("multi-class-val.txt"):
        emo_va = load_emotion("multi-class-val.txt")
    else:                                                             # 검증 파일이 없으면 학습 데이터에서 2,000건 분리
        from sklearn.model_selection import train_test_split
        emo_tr, emo_va = train_test_split(emo_tr, test_size=2000, stratify=emo_tr.label, random_state=42)
else:                                                                 # 파일이 없으면 Hugging Face에서 (같은 데이터)
    from datasets import load_dataset
    ds = load_dataset("dair-ai/emotion")
    names = ds["train"].features["label"].names
    emo_tr = pd.DataFrame({"text": ds["train"]["text"], "label": [names[i] for i in ds["train"]["label"]]})
    emo_va = pd.DataFrame({"text": ds["validation"]["text"], "label": [names[i] for i in ds["validation"]["label"]]})
classes = sorted(emo_tr.label.unique())
print(len(emo_tr), len(emo_va), emo_tr.label.value_counts().to_dict())

e_cnt = Counter(w for s in emo_tr.text for w in s.split())
e_itos = ["<pad>", "<unk>"] + [w for w, c in e_cnt.items() if c >= 2]
e_stoi = {w: i for i, w in enumerate(e_itos)}
def e_encode(texts, max_len=40):
    ids = [[e_stoi.get(w, 1) for w in t.split()][:max_len] or [1] for t in texts]
    X = torch.zeros(len(ids), max_len, dtype=torch.long)
    for i, s in enumerate(ids): X[i, :len(s)] = torch.tensor(s)
    return X, torch.tensor([len(s) for s in ids])
eXtr, eLtr = e_encode(emo_tr.text); eytr = torch.tensor([classes.index(c) for c in emo_tr.label])
eXva, eLva = e_encode(emo_va.text); eyva = torch.tensor([classes.index(c) for c in emo_va.label])

seed_all(42)
emo_model = TextRNN(len(e_itos), cell="lstm", bi=True, n_cls=len(classes))
fit(emo_model, eXtr, eLtr, eytr, epochs=8, bs=64, verbose=False)
emo_acc = evaluate(emo_model, eXva, eLva, eyva)
print(f"감정 6분류 BiLSTM 검증 정확도: {emo_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
emo_model.eval()
with torch.no_grad():
    pred = emo_model(eXva.to(device), eLva).argmax(1).cpu()
print("macro F1:", round(f1_score(eyva, pred, average="macro"), 4), "(4차시: 불균형 데이터는 F1도 함께 본다)")
fig, ax = plt.subplots(figsize=(6.2, 5.2))
ConfusionMatrixDisplay(confusion_matrix(eyva, pred), display_labels=classes).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"감정 6분류 BiLSTM 혼동 행렬 (정확도 {emo_acc:.3f})"); plt.tight_layout()
plt.savefig("figs/emotion_cm.png", dpi=160); plt.show()

In [ ]:
for s in ["i feel so happy that my friends came to my birthday party",
          "i feel nervous about the final exam tomorrow",
          "i can not believe he lied to me again i am so annoyed"]:
    X, L = e_encode([s])
    with torch.no_grad():
        p = torch.softmax(emo_model(X.to(device), L), 1)[0].cpu()
    print(f"{classes[p.argmax()]:<9} {p.max():.2f} | {s}")

---
## **3-5. 데이터가 5,000건뿐이라면?** — Word2Vec 사전학습 임베딩으로 초기화
라벨 없는 대량 텍스트로 먼저 "단어 의미"를 배우고(사전학습), 적은 라벨로 과제를 학습한다 → 오늘날 LLM 미세조정의 원형

In [ ]:
from gensim.models import Word2Vec
# 누수 방지: 테스트 문장은 빼고, 학습 문장 15만 건(라벨은 사용하지 않음)으로만 Word2Vec 학습
w2v = Word2Vec(tok_tr, vector_size=100, window=5, min_count=3, sg=1, negative=5, epochs=5, workers=4, seed=42)
W = torch.randn(len(itos), 100) * 0.1
hit = 0
for i, w in enumerate(itos):
    if w in w2v.wv:
        W[i] = torch.tensor(w2v.wv[w]); hit += 1
print(f"어휘 {len(itos):,}개 중 {hit:,}개를 Word2Vec 벡터로 초기화")

In [ ]:
small = []
for n in [2000, 5000, 20000]:
    for init, pre in [("랜덤 초기화", None), ("Word2Vec 초기화", W)]:
        seed_all(0)
        idx = torch.randperm(len(Xtr))[:n]
        m = TextRNN(len(itos), cell="lstm", pretrained=pre)
        fit(m, Xtr[idx], Ltr[idx], ytr[idx], epochs=5, bs=64, verbose=False)
        acc = evaluate(m, Xte, Lte, yte)
        small.append([n, init, acc]); print(f"학습 {n:>6,}건  {init:<12} 테스트 {acc:.4f}")
small_df = pd.DataFrame(small, columns=["학습 데이터 수", "임베딩", "테스트 정확도"]).pivot(index="학습 데이터 수", columns="임베딩", values="테스트 정확도")
small_df.to_csv("pretrain_effect.csv")
small_df.round(4)

In [ ]:
json.dump(dict(rnn=res, emotion_acc=emo_acc, curves=curves), open("rnn_results.json", "w"), ensure_ascii=False, indent=1)

### 정리
| 모델 | 핵심 | 실험에서 확인한 것 |
|---|---|---|
| RNN | 은닉 상태 + 가중치 공유 | 먼 과거 기울기 급감 |
| LSTM / GRU | 게이트로 기울기 경로 보존 | 기울기 유지, 정확도 향상 |
| BiLSTM | 앞·뒤 문맥을 모두 읽음 | 분류 과제에서 유리 |
| 사전학습 임베딩 | 라벨 없는 데이터로 단어 의미 학습 | 데이터가 적을수록 효과 큼 |